# 002_images.ipynb — Claude en Vertex AI

Imagen real enviada como bytes base64; usa images/prop3.png o IMAGE_PATH.

La primera celda configura el cliente y las rutas. Las llamadas requieren un proyecto y credenciales Google reales. [Configuración](../README.md).

In [ ]:
# Cliente real de Vertex AI: Application Default Credentials (ADC).
import os
from pathlib import Path

from anthropic import AnthropicVertex
from dotenv import load_dotenv

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "GOOGLE" / "README.md").is_file()
)
google_root = repo_root / "GOOGLE"
load_dotenv(google_root / ".env", override=False)
exercise_dir = google_root / "FeatsOfClaude"
os.chdir(exercise_dir)

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
if not project_id:
    raise ValueError("Configura GOOGLE_CLOUD_PROJECT en GOOGLE/.env.")
client = AnthropicVertex(
    project_id=project_id,
    region=os.getenv("CLOUD_ML_REGION", "global"),
)
model = os.getenv("VERTEX_MODEL_ID", "claude-sonnet-4-6")


In [ ]:
# Conserva todos los bloques de las respuestas, incluidas herramientas y thinking.
from anthropic.types import Message


def message_content(message):
    content = message.content if isinstance(message, Message) else message
    if isinstance(content, str):
        if not content.strip():
            raise ValueError("El texto del mensaje no puede estar vacío.")
    elif not isinstance(content, list) or not content:
        raise ValueError("El mensaje necesita bloques de contenido.")
    else:
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text" and not block.get("text", "").strip():
                raise ValueError("Los bloques de texto no pueden estar vacíos.")
    return content


def add_user_message(messages, message):
    messages.append({"role": "user", "content": message_content(message)})


def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message_content(message)})


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


def chat(
    messages,
    system=None,
    stop_sequences=None,
    tools=None,
    tool_choice=None,
    max_tokens=4000,
):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if stop_sequences:
        params["stop_sequences"] = list(stop_sequences)
    if system:
        params["system"] = system
    if tools:
        params["tools"] = list(tools)
        if tool_choice:
            params["tool_choice"] = tool_choice
    message = client.messages.create(**params)
    return message


In [ ]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [ ]:
import base64
import mimetypes
image_path = Path(os.getenv("IMAGE_PATH", str(exercise_dir / "images" / "prop3.png"))).expanduser()
media_type = mimetypes.guess_type(image_path.name)[0]
if media_type not in {"image/png", "image/jpeg", "image/gif", "image/webp"}:
    raise ValueError("La imagen debe ser PNG, JPEG, GIF o WebP.")
image_data = base64.b64encode(image_path.read_bytes()).decode("ascii")
messages = []
add_user_message(messages, [
    {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": image_data}},
    {"type": "text", "text": prompt},
])
response = chat(messages)
text_from_message(response)
